В основном вся нормативная документация выгружена из КонсультанПлюс и в каждом документе присутствует структура.
Виды нормативной документации:
1.	Федеральный закон
2.	ГОСТ
3.	Постановление Правительства РФ
4.	Приказ Министерства промышленности и торговли
5. и т.д.

In [ ]:
#установка библиотеки docx

In [4]:
!pip install python-docx

In [18]:
import re
import requests
import os
import io
import tempfile
from docx import Document
from docx.oxml.ns import nsdecls
from docx.oxml import parse_xml

In [ ]:
#Код для Apps Script для получения списка файлов и их ID
'''
function listFilesInFolder() {
  var folder = DriveApp.getFolderById('1kldI9-mNiesJ4Tj7MQyOiLyzfO0ujVzK');
  var files = folder.getFiles();

  // Создаем новый txt-файл
  var file = DriveApp.createFile('FileList.txt', '');

  // Получаем доступ к содержимому файла
  var fileContent = file.getBlob().getDataAsString();

  while (files.hasNext()) {
    var fileInFolder = files.next();
    // Записываем информацию о файле в txt-файл
    fileContent += fileInFolder.getName() + ': ' + fileInFolder.getId() + '\n';
  }

  // Записываем обновленное содержимое обратно в файл
  file.setContent(fileContent);
}
'''

In [6]:
# Перечень нормативной документации сохранен в файле FileList.txt, его ID 1xlBmSKGs1Wi5ti5xjgKGEIewggvknRxH
id_FileList_txt = '1xlBmSKGs1Wi5ti5xjgKGEIewggvknRxH'

In [7]:
# функция для загрузки документа по doc_id из гугл драйв в текстовом формате
def load_doc_id_text(file_id):
    # Download the document as plain text
    response = requests.get(f'https://drive.google.com/uc?export=download&id={file_id}')
    response.raise_for_status()
    text = response.text

    return text

In [9]:
# Выгрузим перечень нормативной документации
data_txt_id= load_doc_id_text(id_FileList_txt)
data_txt_id

'Постановление 1847.docx: 16XfjaU5Nie7NldFiDQGgPHcFrzqlVJKh\nСОГЛАШЕНИЕ Бурабай 2015.docx: 1_u0BBIJphrq2qBsqog5H0B9OgZjWO2bt\n162-ФЗ.docx: 1zQ_Xn771EuWl9-VZDAiQy2IThdpdgLys\nПриказ МПТ 971.docx: 15J5AZZvjRe_vCYze44tB-fk3amL-2YM0\nПриказ РСТ 2173.docx: 1r4wUVjTqWtNyB-rmI76DEsimSyBIqGlF\nПриказ РСТ 2346.docx: 1nJdos1CgVTcbtQtFhorMoxFPBx6BBgoe\nПостановление 521.docx: 1F17SNGzRYBiHNVLbB6DK6RlVx5LSbeRN\nПМГ 06.2019.docx: 1CxtJO4dVKybY1swOkcXP4buJt1MYoBgb\nРМГ 29.2013.docx: 1qHfddurJ1QEc78sdAeHlVjBL3pflmnbJ\nПриказ МПТ 2907.docx: 1b30hYkfRiqO8TJVZpBwn5ui_RypiJxk0\nПриказ МПТ 456.docx: 1y_2d-GSEctG2q52Pu46fZ-L0cBppAL_y\nПриказ МПТ 4091.docx: 1R5pq1BhN59DXgx2MK-7sH0QbzJN_opWm\nПриказ МПТ 2167.docx: 1zuX2wQqoGKKl5eF9m1PuDTEqnQj9Z-1p\nПриказ МПТ 2906.docx: 1LCNEKwyZDQujUJgMHFdXL9x_kdbkwxoj\nПостановление 734.docx: 1qCFfa--cQb4crDu2uSCytUQcxsx7LlyJ\nПостановление 1053.docx: 1C0lbveUI49b9DVJWzaeTXvTkRFpyssfn\nПостановление 879.docx: 1S1oxQAb98MZJH-zb5JNMbDgoDlgjlLqA\nПостановление 311.docx: 1opVc

In [10]:
#Cохраняем название файлов, данные ID в отдельный библиотеку python
# Создаем пустой словарь
file_dict = {}

# Используем регулярные выражения для поиска всех файлов и их ID
for file, id in re.findall(r'(.*): (.*)', data_txt_id):
    file_dict[file.strip()] = id.strip()

# Выводим словарь
file_dict

{'Постановление 1847.docx': '16XfjaU5Nie7NldFiDQGgPHcFrzqlVJKh',
 'СОГЛАШЕНИЕ Бурабай 2015.docx': '1_u0BBIJphrq2qBsqog5H0B9OgZjWO2bt',
 '162-ФЗ.docx': '1zQ_Xn771EuWl9-VZDAiQy2IThdpdgLys',
 'Приказ МПТ 971.docx': '15J5AZZvjRe_vCYze44tB-fk3amL-2YM0',
 'Приказ РСТ 2173.docx': '1r4wUVjTqWtNyB-rmI76DEsimSyBIqGlF',
 'Приказ РСТ 2346.docx': '1nJdos1CgVTcbtQtFhorMoxFPBx6BBgoe',
 'Постановление 521.docx': '1F17SNGzRYBiHNVLbB6DK6RlVx5LSbeRN',
 'ПМГ 06.2019.docx': '1CxtJO4dVKybY1swOkcXP4buJt1MYoBgb',
 'РМГ 29.2013.docx': '1qHfddurJ1QEc78sdAeHlVjBL3pflmnbJ',
 'Приказ МПТ 2907.docx': '1b30hYkfRiqO8TJVZpBwn5ui_RypiJxk0',
 'Приказ МПТ 456.docx': '1y_2d-GSEctG2q52Pu46fZ-L0cBppAL_y',
 'Приказ МПТ 4091.docx': '1R5pq1BhN59DXgx2MK-7sH0QbzJN_opWm',
 'Приказ МПТ 2167.docx': '1zuX2wQqoGKKl5eF9m1PuDTEqnQj9Z-1p',
 'Приказ МПТ 2906.docx': '1LCNEKwyZDQujUJgMHFdXL9x_kdbkwxoj',
 'Постановление 734.docx': '1qCFfa--cQb4crDu2uSCytUQcxsx7LlyJ',
 'Постановление 1053.docx': '1C0lbveUI49b9DVJWzaeTXvTkRFpyssfn',
 'Постано

In [27]:
# функция для загрузки документа по docx_id из гугл драйв в формате docx
def read_docx_from_id(docx_id):
    response = requests.get(f'https://docs.google.com/uc?export=download&id={docx_id}')
    file = io.BytesIO(response.content)
    document = Document(file)
    return document


In [13]:
def print_paragraph_style(document):
    styles = set()
    for paragraph in document.paragraphs:
        styles.add(paragraph.style.name)
    return styles

In [28]:
#Пройдемся по всем документам и сохраним применяемые стили
style_dict = {}
for file_name, file_id in file_dict.items():
    doc = read_docx_from_id(file_id)  # открываем документ
    style = print_paragraph_style(doc)  # применяем функцию к документу
    style_dict[file_name] = style

In [29]:
for file_name, style in style_dict.items():
  print (file_name, style)

Постановление 1847.docx {'ConsPlusTitlePage', 'Normal', 'ConsPlusNormal', 'ConsPlusTitle'}
СОГЛАШЕНИЕ Бурабай 2015.docx {'Normal'}
162-ФЗ.docx {'ConsPlusTitlePage', 'Normal', 'ConsPlusNormal', 'ConsPlusTitle'}
Приказ МПТ 971.docx {'ConsPlusTitlePage', 'ConsPlusNonformat', 'ConsPlusNormal', 'Normal', 'ConsPlusTitle'}
Приказ РСТ 2173.docx {'ConsPlusTitlePage', 'ConsPlusNonformat', 'ConsPlusNormal', 'Normal', 'ConsPlusTitle'}
Приказ РСТ 2346.docx {'ConsPlusTitlePage', 'ConsPlusNonformat', 'ConsPlusNormal', 'Normal', 'ConsPlusTitle'}
Постановление 521.docx {'ConsPlusTitlePage', 'Normal', 'ConsPlusNormal', 'ConsPlusTitle'}
ПМГ 06.2019.docx {'ConsPlusTitlePage', 'Normal', 'ConsPlusNormal', 'ConsPlusTitle'}
РМГ 29.2013.docx {'ConsPlusTitlePage', 'Normal', 'ConsPlusNormal', 'ConsPlusTitle'}
Приказ МПТ 2907.docx {'ConsPlusTitlePage', 'Normal', 'ConsPlusNormal', 'ConsPlusTitle'}
Приказ МПТ 456.docx {'ConsPlusTitlePage', 'ConsPlusNonformat', 'ConsPlusNormal', 'Normal', 'ConsPlusTitle'}
Приказ МПТ

Почти в каждом нормативном документе применяются одинаковые стили: ConsPlusTitlePage', 'ConsPlusTitle', 'Normal', 'ConsPlusNormal

Стандартными средствами выводятся только стили параграфов

#Объединение параграфов

In [30]:
#Добавлен уровень пунктов
def preprocess_res_dub3(document):
    markdown_lines = []  # список для хранения строк в формате Markdown
    current_paragraph = ""  # текущий параграф
    current_header_level = 1  # текущий уровень заголовка
    count_styles = 0  # счетчик стилей
    reset_flag = False  # флаг для сброса

    # проходим по каждому параграфу в документе
    for i, paragraph in enumerate(document.paragraphs):
        markdown_text = paragraph.text  # получаем текст параграфа

        # если текст параграфа не пустой
        if markdown_text:
            p_xml = paragraph._p.xml  # получаем xml параграфа
            root = parse_xml(r'{}'.format(p_xml))  # парсим xml
            namespaces = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}  # пространство имен для xml
            style = root.find('.//w:pPr/w:pStyle', namespaces=namespaces)  # ищем стиль параграфа
            outline_lvl = root.find('.//w:pPr/w:outlineLvl', namespaces=namespaces)  # ищем уровень заголовка

            # если уровень заголовка определен
            if outline_lvl is not None:
                outline_lvl_val = int(outline_lvl.get('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val'))  # получаем значение уровня заголовка
                if current_header_level != outline_lvl_val + 1: count_styles = 0  # если текущий уровень заголовка не равен полученному, сбрасываем счетчик стилей
                current_header_level = outline_lvl_val + 1  # обновляем текущий уровень заголовка

                # если стиль параграфа определен
                if style is not None:
                    style_val = style.get('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val')  # получаем значение стиля
                    if style_val == 'ConsPlusTitle':  # если стиль равен 'ConsPlusTitle'
                        reset_flag = False  # сбрасываем флаг
                        count_styles += 1  # увеличиваем счетчик стилей
                        if count_styles == 1: current_paragraph = "#" * current_header_level + " " + markdown_text  # если счетчик стилей равен 1, формируем заголовок
                        elif count_styles > 1:  # если счетчик стилей больше 1
                            current_paragraph = current_paragraph + " " + markdown_text  # добавляем текст к текущему параграфу
                    else:  # если стиль не равен 'ConsPlusTitle'
                        count_styles = 0  # сбрасываем счетчик стилей
                        if not reset_flag:  # если флаг не установлен
                            markdown_lines.append(current_paragraph + "\n")  # добавляем текущий параграф в список строк
                            markdown_lines.append(current_paragraph + "\n")  # добавляем текущий параграф в список строк еще раз
                            reset_flag = True  # устанавливаем флаг
                        markdown_lines.append(markdown_text + "\n")  # добавляем текст параграфа в список строк

            else:  # если уровень заголовка не определен
                style_val = style.get('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val')  # получаем значение стиля
                if style_val == 'ConsPlusTitle':  # если стиль равен 'ConsPlusTitle'
                    reset_flag = False  # сбрасываем флаг
                    count_styles += 1  # увеличиваем счетчик стилей
                    if count_styles == 1: current_paragraph = "#" * current_header_level + " " + markdown_text  # если счетчик стилей равен 1, формируем заголовок
                    elif count_styles > 1:  # если счетчик стилей больше 1
                        current_paragraph = current_paragraph + " " + markdown_text  # добавляем текст к текущему параграфу
                else:  # если стиль не равен 'ConsPlusTitle'
                    count_styles = 0  # сбрасываем счетчик стилей
                    if not reset_flag:  # если флаг не установлен
                        markdown_lines.append(current_paragraph + "\n")  # добавляем текущий параграф в список строк
                        markdown_lines.append(current_paragraph + "\n")  # добавляем текущий параграф в список строк еще раз
                        reset_flag = True  # устанавливаем флаг
                    match = re.match(r'^\d+\.', markdown_text) # поиск пунктов 1., 2.
                    if match:
                        markdown_text = '\n'+ '#' * (current_header_level + 1) + ' пункт ' + match.group()+'\n' + markdown_text
                        markdown_lines.append(markdown_text + "\n")  # добавляем новый текст параграфа в список строк
                    else: markdown_lines.append(markdown_text + "\n")  # добавляем текст параграфа в список строк

    return markdown_lines  # возвращаем список строк в формате Markdown

#Объединим все документы в одну базу знаний

In [31]:
#Пройдемся по всем документам и применим функцию обработки для документов у которых есть стиль ConsPlusTitle
style_dict = {}
doc_markdown = ''
for file_name, file_id in file_dict.items():
    doc = read_docx_from_id(file_id)  # открываем документ
    styles = print_paragraph_style(doc)  # применяем функцию к документу
    if 'ConsPlusTitle' in styles: #если в документе есть стиль ConsPlusTitle, проводим его предобработку и добавляем в общую базу
        doc_markdown += ''.join(preprocess_res_dub3(doc))
with open('v3.0_doc_markdown.txt', 'a') as f: #сохраним в файл
    f.write(doc_markdown )

Объединеннная и сохраненная база нормативной документации V3.0 сохранена на гугл диск по ссылке:
https://drive.google.com/file/d/1JEtc7cLvqt0Qa5i3LbbHwtL8lFjeFmbk/view?usp=sharing